In [4]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [6]:
df = pd.read_csv(INPUT_DIR / "Manhattan! - M_1200_INDICATORS_6.csv")
df.head(1208)

,Unnamed: 0,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,1,o_360610002011,1294.883739,1067.296,0,19.750000,18.116667,2.100000
1,2,o_360610002012,1887.846508,1125.124,0,24.833333,19.083333,2.050000
2,3,o_360610002021,2267.091673,1076.668,0,23.033333,18.250000,2.316667
3,4,o_360610002022,2477.358670,1112.161,0,24.200000,18.850000,1.150000
4,5,o_360610002023,2339.396673,1148.973,0,24.283333,19.500000,1.066667
...,...,...,...,...,...,...,...,...
1202,1203,o_360610317041,1789.744027,1221.106,0,22.566667,21.066667,1.116667
1203,1204,o_360610317042,2006.648553,1528.198,0,27.950000,26.950000,2.950000
1204,1205,o_360610317043,1953.957876,1015.330,0,20.483333,17.483333,1.200000
1205,1206,o_360610317044,2256.353506,960.722,0,20.600000,16.600000,1.083333


In [7]:
print(df.shape)
print(df.columns.tolist())

(1207, 8)
['Unnamed: 0', 'from_id', 'total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [8]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(1208)

,from_id,GEOID_TEXT
0,o_360610002011,360610002011
1,o_360610002012,360610002012
2,o_360610002021,360610002021
3,o_360610002022,360610002022
4,o_360610002023,360610002023
...,...,...
1202,o_360610317041,360610317041
1203,o_360610317042,360610317042
1204,o_360610317043,360610317043
1205,o_360610317044,360610317044


In [9]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",


    "wait_time_total"]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [12]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,320.034559,8289.273639
walking_distance,108.267000,2639.960000
transfers,0.000000,2.000000
travel_time_total,3.233333,48.900000
walking_time,1.783333,44.400000
wait_time_total,1.016667,14.983333


In [13]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.877673,0.621191,1.0,0.638321,0.616738,0.922434
1,0.803267,0.598349,1.0,0.527007,0.594056,0.926014
2,0.755678,0.617489,1.0,0.566423,0.613610,0.906921
3,0.729294,0.603469,1.0,0.540876,0.599531,0.990453
4,0.746605,0.588929,1.0,0.539051,0.584278,0.996420


In [14]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance       1033.051890
walking_distance      856.474057
transfers            1166.500000
travel_time_total     847.737956
walking_time          852.742276
wait_time_total      1013.261336
dtype: float64

In [15]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000850,0.000725,0.000857,0.000753,0.000723,0.000910
1,0.000778,0.000699,0.000857,0.000622,0.000697,0.000914
2,0.000732,0.000721,0.000857,0.000668,0.000720,0.000895
3,0.000706,0.000705,0.000857,0.000638,0.000703,0.000977
4,0.000723,0.000688,0.000857,0.000636,0.000685,0.000983


In [16]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
dtype: float64

In [17]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000850,0.000725,0.000857,0.000753,0.000723,0.000910
1,0.000778,0.000699,0.000857,0.000622,0.000697,0.000914
2,0.000732,0.000721,0.000857,0.000668,0.000720,0.000895
3,0.000706,0.000705,0.000857,0.000638,0.000703,0.000977
4,0.000723,0.000688,0.000857,0.000636,0.000685,0.000983


In [18]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-7.070753,-7.228941,-7.061763,-7.191485,-7.231768,-7.001669
1,-7.159341,-7.266405,-7.061763,-7.383112,-7.269240,-6.997795
2,-7.220412,-7.234918,-7.061763,-7.310985,-7.236854,-7.018629
3,-7.255952,-7.257884,-7.061763,-7.357137,-7.260065,-6.930522
4,-7.232491,-7.282274,-7.061763,-7.360516,-7.285835,-6.924516


In [19]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-0.006007,-0.005243,-0.006054,-0.005415,-0.005230,-0.006374
1,-0.005567,-0.005076,-0.006054,-0.004590,-0.005064,-0.006395
2,-0.005282,-0.005216,-0.006054,-0.004885,-0.005207,-0.006282
3,-0.005122,-0.005114,-0.006054,-0.004694,-0.005104,-0.006775
4,-0.005227,-0.005007,-0.006054,-0.004680,-0.004992,-0.006809
5,-0.005047,-0.005868,-0.006054,-0.005217,-0.005861,-0.005398
6,-0.004431,-0.004345,-0.006054,-0.004275,-0.004338,-0.004786
7,-0.004858,-0.005928,-0.003324,-0.005053,-0.005919,-0.005405


In [20]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -7.088597
walking_distance    -7.073313
transfers           -7.085234
travel_time_total   -7.074092
walking_time        -7.072699
wait_time_total     -7.078422
dtype: float64

In [21]:
n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 1207
k value: 0.14092658511641593
Step 5: Entropy for each indicator


total_distance       0.998972
walking_distance     0.996818
transfers            0.998498
travel_time_total    0.996928
walking_time         0.996731
wait_time_total      0.997538
dtype: float64

In [22]:
#df["fare"].value_counts()

In [23]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.001028
walking_distance     0.003182
transfers            0.001502
travel_time_total    0.003072
walking_time         0.003269
wait_time_total      0.002462
dtype: float64

In [24]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.070833
walking_distance     0.219226
transfers            0.103482
travel_time_total    0.211656
walking_time         0.225180
wait_time_total      0.169623
dtype: float64

In [25]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.998972,0.001028,0.070833
walking_distance,0.996818,0.003182,0.219226
transfers,0.998498,0.001502,0.103482
travel_time_total,0.996928,0.003072,0.211656
walking_time,0.996731,0.003269,0.225180
wait_time_total,0.997538,0.002462,0.169623


In [26]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
walking_time,0.996731,0.003269,0.225180
walking_distance,0.996818,0.003182,0.219226
travel_time_total,0.996928,0.003072,0.211656
wait_time_total,0.997538,0.002462,0.169623
transfers,0.998498,0.001502,0.103482
total_distance,0.998972,0.001028,0.070833


In [27]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360610002011,360610002011,0.732279
1,o_360610002012,360610002012,0.693941
2,o_360610002021,360610002021,0.704273
3,o_360610002022,360610002022,0.704922
4,o_360610002023,360610002023,0.700152
5,o_360610002024,360610002024,0.726327
6,o_360610002025,360610002025,0.583444
7,o_360610002026,360610002026,0.671564
8,o_360610005001,360610005001,0.474197
9,o_360610005002,360610005002,0.464437


In [28]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    1207.000000
mean        0.766337
std         0.102785
min         0.159093
25%         0.706957
50%         0.774491
75%         0.844176
max         0.959748
Name: ewm_accessibility_score, dtype: float64

In [29]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,ewm_accessibility_score
0,360610002011,o_360610002011,1294.883739,1067.296,0,19.750000,18.116667,2.100000,0.732279
1,360610002012,o_360610002012,1887.846508,1125.124,0,24.833333,19.083333,2.050000,0.693941
2,360610002021,o_360610002021,2267.091673,1076.668,0,23.033333,18.250000,2.316667,0.704273
3,360610002022,o_360610002022,2477.358670,1112.161,0,24.200000,18.850000,1.150000,0.704922
4,360610002023,o_360610002023,2339.396673,1148.973,0,24.283333,19.500000,1.066667,0.700152
5,360610002024,o_360610002024,2576.681020,847.335,0,20.983333,14.400000,4.366667,0.726327
6,360610002025,o_360610002025,3375.847529,1374.579,0,26.733333,23.233333,5.750000,0.583444
7,360610002026,o_360610002026,2823.592450,826.163,1,22.000000,14.050000,4.350000,0.671564
8,360610005001,o_360610005001,5193.439474,1461.789,1,32.366667,25.033333,5.600000,0.474197
9,360610005002,o_360610005002,4234.746951,1960.732,0,37.233333,33.233333,2.383333,0.464437


In [31]:
final_results.to_csv(OUTPUT_DIR / "manhattan_ewm_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "manhattan_ewm_weights.csv", index=True)

print("Saved manhattan_ewm_results.csv")
print("Saved manhattan_ewm_weights.csv")

Saved manhattan_ewm_results.csv
Saved manhattan_ewm_weights.csv
